# 01 — Data Collection & First Inspection

**Project:** Customer Churn Prediction
**Goal of this notebook:** Load the Telco dataset and understand its structure
before touching anything.

**Questions I want answered:**
1. How many customers and columns do we have?
2. What does each column mean?
3. Are there missing values or wrong data types?
4. What does the churn split look like?

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
print("Setup complete")

Setup complete


## 1. Load the data

Loading the raw CSV using a relative path so this notebook runs on any machine.

In [3]:
file_path = "../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"

df = pd.read_csv(file_path)

print("Data loaded successfully")
print("Shape (rows, columns):", df.shape)

Data loaded successfully
Shape (rows, columns): (7043, 21)


## 2. First look at the data

Checking the first rows and the column structure.

In [4]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 3. Data types and missing values

Checking that every column is stored as the right type.

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

## 4. How many customers churned?

The target variable's distribution tells us how balanced the problem is.

In [6]:
print(df["Churn"].value_counts())
print()
print(df["Churn"].value_counts(normalize=True).round(3) * 100)

Churn
No     5174
Yes    1869
Name: count, dtype: int64

Churn
No     73.5
Yes    26.5
Name: proportion, dtype: float64


## 5. Data quality: investigating TotalCharges

TotalCharges should be numeric but is stored as text. Finding out why.

In [7]:
blank_mask = df["TotalCharges"].str.strip() == ""

print("Rows with blank TotalCharges:", blank_mask.sum())

df.loc[blank_mask, ["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]]

Rows with blank TotalCharges: 11


,customerID,tenure,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,0,52.55,,No
753,3115-CZMZD,0,20.25,,No
936,5709-LVOEQ,0,80.85,,No
1082,4367-NUYAO,0,25.75,,No
1340,1371-DWPAZ,0,56.05,,No
3331,7644-OMVMY,0,19.85,,No
3826,3213-VVOLG,0,25.35,,No
4380,2520-SGTTA,0,20.00,,No
5218,2923-ARZLG,0,19.70,,No
6670,4075-WKNIU,0,73.35,,No


## 6. Fixing TotalCharges

Converting to numeric. Blank entries become NaN (missing), which I then handle explicitly.

In [8]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

print("New dtype:", df["TotalCharges"].dtype)
print("Missing values:", df["TotalCharges"].isnull().sum())

New dtype: float64
Missing values: 11


## 7. Handling the 11 missing TotalCharges values

All 11 rows have tenure = 0, meaning brand-new customers who have not been billed yet.
TotalCharges is therefore genuinely 0, not unknown. Filling with 0 reflects reality;
filling with the mean would invent spending that never happened.

In [9]:
df["TotalCharges"] = df["TotalCharges"].fillna(0)

print("Missing values left:", df["TotalCharges"].isnull().sum())
print("Rows with TotalCharges = 0:", (df["TotalCharges"] == 0).sum())
print("Do they all have tenure 0?", (df.loc[df["TotalCharges"] == 0, "tenure"] == 0).all())

Missing values left: 0
Rows with TotalCharges = 0: 11
Do they all have tenure 0? True


## 8. Duplicates and summary statistics

Checking that each row is a unique customer and getting a feel for numeric columns.

In [10]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate customer IDs:", df["customerID"].duplicated().sum())

df.describe()

Duplicate rows: 0
Duplicate customer IDs: 0


,SeniorCitizen,tenure,MonthlyCharges,TotalCharges
count,7043.000000,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692,2279.734304
std,0.368612,24.559481,30.090047,2266.794470
min,0.000000,0.000000,18.250000,0.000000
25%,0.000000,9.000000,35.500000,398.550000
50%,0.000000,29.000000,70.350000,1394.550000
75%,0.000000,55.000000,89.850000,3786.600000
max,1.000000,72.000000,118.750000,8684.800000


## 9. Save the cleaned data

Saving to data/processed/ so the next notebook starts from this cleaned version
without re-running the cleaning steps.

In [11]:
import os

os.makedirs("../data/processed", exist_ok=True)

df.to_csv("../data/processed/telco_clean.csv", index=False)

print("Saved:", df.shape)

Saved: (7043, 21)


## Summary of findings

- 7,043 customers, 21 columns, no duplicates
- Target: `Churn` (about 26.5% Yes, so the classes are imbalanced)
- `TotalCharges` was stored as text; 11 blanks, all with tenure = 0
  (new customers not yet billed), filled with 0
- `customerID` is an identifier only and will be dropped before modelling
- Hypothesis to test in EDA: short tenure and month-to-month contracts drive churn